In [16]:
# ── Cell 1 — Imports & Load Files ──────────────────────────────────────────

import pandas as pd
from google.colab import files

# ── Upload your 3 files when prompted ──────────────────────────────────────
uploaded = files.upload()

# ── Load ───────────────────────────────────────────────────────────────────
f1 = pd.read_excel('plant_identification_v2.xlsx')
f2 = pd.read_csv('disease_treatment_english_v1.csv')
f3 = pd.read_excel('final_disease_ontology.xlsx')

# ── Confirm ────────────────────────────────────────────────────────────────
print(f'File 1 (herbs):      {f1.shape[0]} rows, {f1.shape[1]} cols')
print(f'File 2 (claims):     {f2.shape[0]} rows, {f2.shape[1]} cols')
print(f'File 3 (clustering): {f3.shape[0]} rows, {f3.shape[1]} cols')

Saving plant_identification_v2.xlsx to plant_identification_v2.xlsx
File 1 (herbs):      645 rows, 11 cols
File 2 (claims):     7362 rows, 8 cols
File 3 (clustering): 3577 rows, 8 cols


In [17]:
# ── Cell 2 — Select Columns ────────────────────────────────────────────────

f1 = f1[['entry_id', 'scientific_name', 'gbif_validated', 'other_known_names']]

f2 = f2[['entry_id', 'arabic_name', 'disease_or_condition_en',
          'treatment_method_en', 'expected_effect_en']]

f3 = f3[['normalized_disease', 'cluster_label', 'broader_category']]

# ── Confirm ────────────────────────────────────────────────────────────────
print(f'File 1 columns: {f1.columns.tolist()}')
print(f'File 2 columns: {f2.columns.tolist()}')
print(f'File 3 columns: {f3.columns.tolist()}')

File 1 columns: ['entry_id', 'scientific_name', 'gbif_validated', 'other_known_names']
File 2 columns: ['entry_id', 'arabic_name', 'disease_or_condition_en', 'treatment_method_en', 'expected_effect_en']
File 3 columns: ['normalized_disease', 'cluster_label', 'broader_category']


In [18]:
# ── Cell 3 — Pre-filter File 2 ─────────────────────────────────────────────

before = len(f2)

f2 = f2[f2['treatment_method_en'].notna()].reset_index(drop=True)

after = len(f2)

# ── Confirm ────────────────────────────────────────────────────────────────
print(f'Rows before filter: {before}')
print(f'Rows removed:       {before - after}')
print(f'Rows remaining:     {after}')

Rows before filter: 7362
Rows removed:       31
Rows remaining:     7331


In [19]:
# ── Cell 4 — Join File 2 to File 3 ─────────────────────────────────────────

# Prepare join keys
f2['join_key'] = f2['disease_or_condition_en'].str.strip().str.lower()
f3['join_key'] = f3['normalized_disease'].str.strip().str.lower()

# Join
f2 = f2.merge(f3[['join_key', 'cluster_label', 'broader_category']],
              on='join_key',
              how='left')

# Drop join key — no longer needed
f2 = f2.drop(columns=['join_key'])

# ── Validate ───────────────────────────────────────────────────────────────
unmatched = f2['cluster_label'].isna().sum()

print(f'Total claims after join: {len(f2)}')
print(f'Claims with cluster:     {f2["cluster_label"].notna().sum()}')
print(f'Claims without cluster:  {unmatched}')

if unmatched > 0:
    print('\nWARNING — unmatched diseases:')
    print(f2[f2['cluster_label'].isna()]['disease_or_condition_en'].unique())
else:
    print('\n✓ All claims matched to a cluster')

Total claims after join: 7331
Claims with cluster:     7331
Claims without cluster:  0

✓ All claims matched to a cluster


In [20]:
# ── Cell 5 — Assign claim_id ───────────────────────────────────────────────

f2.insert(0, 'claim_id', [f'CLAIM_{i+1:05d}' for i in range(len(f2))])

# ── Confirm ────────────────────────────────────────────────────────────────
print(f'Total claims: {len(f2)}')
print(f'First 3 claim_ids: {f2["claim_id"].head(3).tolist()}')
print(f'Last 3 claim_ids:  {f2["claim_id"].tail(3).tolist()}')
print(f'Duplicates in claim_id: {f2["claim_id"].duplicated().sum()}')
print()
print(f2[['claim_id', 'entry_id', 'disease_or_condition_en', 'cluster_label']].head(5).to_string())

Total claims: 7331
First 3 claim_ids: ['CLAIM_00001', 'CLAIM_00002', 'CLAIM_00003']
Last 3 claim_ids:  ['CLAIM_07329', 'CLAIM_07330', 'CLAIM_07331']
Duplicates in claim_id: 0

      claim_id  entry_id        disease_or_condition_en              cluster_label
0  CLAIM_00001         1       Skin blemishes and scars  dermatological_conditions
1  CLAIM_00002         1           Favus (scalp ulcers)      hair_scalp_conditions
2  CLAIM_00003         1                 Scalp pustules      hair_scalp_conditions
3  CLAIM_00004         1                    Common cold     respiratory_conditions
4  CLAIM_00005         1  Dyspnea (shortness of breath)     respiratory_conditions


In [21]:
# ── Cell 6 — Group into Search Units ──────────────────────────────────────

# Group by entry_id + cluster_label
grouped = f2.groupby(['entry_id', 'cluster_label', 'broader_category']).agg(
    claim_ids_covered=('claim_id', list),
    claims_count=('claim_id', 'count')
).reset_index()

# Assign search_unit_id
grouped.insert(0, 'search_unit_id', [f'SU_{i+1:05d}' for i in range(len(grouped))])

# ── Confirm ────────────────────────────────────────────────────────────────
print(f'Total claims going in:      7331')
print(f'Search units created:       {len(grouped)}')
print(f'Compression ratio:          {7331 / len(grouped):.2f}x')
print(f'Reduction:                  {((7331 - len(grouped)) / 7331 * 100):.1f}%')
print()
print(f'Units covering 1 claim:     {(grouped["claims_count"] == 1).sum()}')
print(f'Units covering 2-5 claims:  {((grouped["claims_count"] >= 2) & (grouped["claims_count"] <= 5)).sum()}')
print(f'Units covering 6+ claims:   {(grouped["claims_count"] >= 6).sum()}')
print()
print('Sample search units:')
print(grouped[['search_unit_id', 'entry_id', 'cluster_label', 'claims_count']].head(8).to_string())

Total claims going in:      7331
Search units created:       5014
Compression ratio:          1.46x
Reduction:                  31.6%

Units covering 1 claim:     3478
Units covering 2-5 claims:  1518
Units covering 6+ claims:   18

Sample search units:
  search_unit_id  entry_id                cluster_label  claims_count
0       SU_00001         1    dermatological_conditions             1
1       SU_00002         1  gastrointestinal_conditions             4
2       SU_00003         1        hair_scalp_conditions             2
3       SU_00004         1         metabolic_conditions             1
4       SU_00005         1   musculoskeletal_conditions             1
5       SU_00006         1      neurological_conditions             1
6       SU_00007         1             renal_conditions             1
7       SU_00008         1       respiratory_conditions             3


In [22]:
# ── Cell 7 — Derive Disease Synonyms ──────────────────────────────────────

# Group all normalized_disease values per cluster_label from File 3
# Every disease in the same cluster becomes a synonym for that cluster

synonyms_map = (
    f3.groupby('cluster_label')['normalized_disease']
    .apply(list)
    .to_dict()
)

# Attach to search units
grouped['disease_synonyms'] = grouped['cluster_label'].map(synonyms_map)

# ── Confirm ────────────────────────────────────────────────────────────────
print(f'Clusters with synonyms: {len(synonyms_map)}')
print()
print('Sample — respiratory_conditions synonyms (first 10):')
print(synonyms_map['respiratory_conditions'][:10])
print()
print('Sample — gastrointestinal_conditions synonyms (first 10):')
print(synonyms_map['gastrointestinal_conditions'][:10])
print()
print(f'Total synonyms per cluster:')
for cluster, syns in sorted(synonyms_map.items(), key=lambda x: len(x[1]), reverse=True)[:8]:
    print(f'  {cluster}: {len(syns)} synonyms')

Clusters with synonyms: 31

Sample — respiratory_conditions synonyms (first 10):
['cough', 'asthma', 'dyspnea', 'chronic cough', 'pleurisy', 'hemoptysis', 'hiccups', 'dyspnea (shortness of breath)', 'common cold', 'pulmonary diseases']

Sample — gastrointestinal_conditions synonyms (first 10):
['hemorrhoids', 'colic', 'diarrhea', 'gastric weakness', 'flatulence', 'flatulence and bloating', 'dyspepsia', 'nausea', 'chronic diarrhea', 'anorexia']

Total synonyms per cluster:
  gastrointestinal_conditions: 341 synonyms
  dermatological_conditions: 312 synonyms
  reproductive_conditions: 252 synonyms
  metabolic_conditions: 237 synonyms
  neurological_conditions: 216 synonyms
  respiratory_conditions: 206 synonyms
  psychological_conditions: 157 synonyms
  ophthalmological_conditions: 149 synonyms


In [23]:
# ── Cell 8 — Enrich Search Units with File 1 ──────────────────────────────

search_units = grouped.merge(
    f1[['entry_id', 'scientific_name', 'gbif_validated', 'other_known_names']],
    on='entry_id',
    how='left'
)

# Reorder columns cleanly
search_units = search_units[[
    'search_unit_id',
    'entry_id',
    'scientific_name',
    'gbif_validated',
    'other_known_names',
    'cluster_label',
    'broader_category',
    'disease_synonyms',
    'claim_ids_covered',
    'claims_count'
]]

# ── Confirm ────────────────────────────────────────────────────────────────
print(f'Search units total: {len(search_units)}')
print(f'Nulls in scientific_name: {search_units["scientific_name"].isna().sum()}')
print(f'Nulls in gbif_validated:  {search_units["gbif_validated"].isna().sum()}')
print()
print('Sample row:')
sample = search_units.iloc[1]
print(f'  search_unit_id:   {sample["search_unit_id"]}')
print(f'  entry_id:         {sample["entry_id"]}')
print(f'  scientific_name:  {sample["scientific_name"]}')
print(f'  gbif_validated:   {sample["gbif_validated"]}')
print(f'  cluster_label:    {sample["cluster_label"]}')
print(f'  broader_category: {sample["broader_category"]}')
print(f'  claims_count:     {sample["claims_count"]}')
print(f'  claim_ids_covered:{sample["claim_ids_covered"]}')
print(f'  synonyms (first 3):{sample["disease_synonyms"][:3]}')

Search units total: 5014
Nulls in scientific_name: 0
Nulls in gbif_validated:  0

Sample row:
  search_unit_id:   SU_00002
  entry_id:         1
  scientific_name:  Ammi majus
  gbif_validated:   True
  cluster_label:    gastrointestinal_conditions
  broader_category: gastrointestinal
  claims_count:     4
  claim_ids_covered:['CLAIM_00007', 'CLAIM_00010', 'CLAIM_00011', 'CLAIM_00012']
  synonyms (first 3):['hemorrhoids', 'colic', 'diarrhea']


In [25]:
# ── Cell 9 — Validate ─────────────────────────────────────────────────────

errors = []

# Check 1: Total search units have no nulls in critical columns
for col in ['search_unit_id', 'entry_id', 'scientific_name', 'cluster_label']:
    nulls = search_units[col].isna().sum()
    if nulls > 0:
        errors.append(f'FAIL — {nulls} nulls found in {col}')
    else:
        print(f'✓ No nulls in {col}')

# Check 2: search_unit_id is unique
dupes = search_units['search_unit_id'].duplicated().sum()
if dupes > 0:
    errors.append(f'FAIL — {dupes} duplicate search_unit_ids found')
else:
    print(f'✓ search_unit_id is unique')

# Check 3: Every claim_id in f2 appears in exactly one search unit
all_covered_ids = [cid for ids in search_units['claim_ids_covered'] for cid in ids]
covered_set = set(all_covered_ids)
f2_set = set(f2['claim_id'])

missing_claims = f2_set - covered_set
extra_claims = covered_set - f2_set

if missing_claims:
    errors.append(f'FAIL — {len(missing_claims)} claims not covered by any search unit')
else:
    print(f'✓ All 7331 claims covered by a search unit')

if extra_claims:
    errors.append(f'FAIL — {len(extra_claims)} unknown claim_ids in search units')
else:
    print(f'✓ No unknown claim_ids in search units')

# Check 4: Sum of claims_count equals total claims
total_from_units = search_units['claims_count'].sum()
if total_from_units != len(f2):
    errors.append(f'FAIL — claims_count sum is {total_from_units}, expected {len(f2)}')
else:
    print(f'✓ claims_count sum matches total claims ({total_from_units})')

# Check 5: claims reference has no nulls in critical columns
claims_reference = f2[['claim_id', 'entry_id', 'arabic_name',
                        'disease_or_condition_en', 'cluster_label',
                        'treatment_method_en', 'expected_effect_en']].copy()

for col in ['claim_id', 'entry_id', 'cluster_label']:
    nulls = claims_reference[col].isna().sum()
    if nulls > 0:
        errors.append(f'FAIL — {nulls} nulls in claims_reference column {col}')
    else:
        print(f'✓ No nulls in claims_reference.{col}')

# ── Final verdict ──────────────────────────────────────────────────────────
print()
if errors:
    print('═══ VALIDATION FAILED ═══')
    for e in errors:
        print(f'  {e}')
else:
    print('═══ ALL CHECKS PASSED — READY TO SAVE ═══')

✓ No nulls in search_unit_id
✓ No nulls in entry_id
✓ No nulls in scientific_name
✓ No nulls in cluster_label
✓ search_unit_id is unique
✓ All 7331 claims covered by a search unit
✓ No unknown claim_ids in search units
✓ claims_count sum matches total claims (7331)
✓ No nulls in claims_reference.claim_id
✓ No nulls in claims_reference.entry_id
✓ No nulls in claims_reference.cluster_label

═══ ALL CHECKS PASSED — READY TO SAVE ═══


In [26]:
# ── Cell 10 — Build Claims Reference & Save Outputs ───────────────────────

# Build claims reference table
claims_reference = f2[[
    'claim_id',
    'entry_id',
    'arabic_name',
    'disease_or_condition_en',
    'cluster_label',
    'treatment_method_en',
    'expected_effect_en'
]].copy()

# Add search_unit_id to claims reference
claim_to_su = {
    cid: row['search_unit_id']
    for _, row in search_units.iterrows()
    for cid in row['claim_ids_covered']
}
claims_reference['search_unit_id'] = claims_reference['claim_id'].map(claim_to_su)

# Reorder columns
claims_reference = claims_reference[[
    'claim_id',
    'search_unit_id',
    'entry_id',
    'arabic_name',
    'disease_or_condition_en',
    'cluster_label',
    'treatment_method_en',
    'expected_effect_en'
]]

# Save both files
search_units.to_excel('search_units.xlsx', index=False)
claims_reference.to_excel('claims_reference.xlsx', index=False)

# Download
from google.colab import files
files.download('search_units.xlsx')
files.download('claims_reference.xlsx')

# ── Final summary ──────────────────────────────────────────────────────────
print('═══ STAGE 2 COMPLETE ═══')
print()
print(f'search_units.xlsx')
print(f'  Rows:    {len(search_units)}')
print(f'  Columns: {search_units.columns.tolist()}')
print()
print(f'claims_reference.xlsx')
print(f'  Rows:    {len(claims_reference)}')
print(f'  Columns: {claims_reference.columns.tolist()}')
print()
print(f'Nulls in search_unit_id (claims_reference): {claims_reference["search_unit_id"].isna().sum()}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

═══ STAGE 2 COMPLETE ═══

search_units.xlsx
  Rows:    5014
  Columns: ['search_unit_id', 'entry_id', 'scientific_name', 'gbif_validated', 'other_known_names', 'cluster_label', 'broader_category', 'disease_synonyms', 'claim_ids_covered', 'claims_count']

claims_reference.xlsx
  Rows:    7331
  Columns: ['claim_id', 'search_unit_id', 'entry_id', 'arabic_name', 'disease_or_condition_en', 'cluster_label', 'treatment_method_en', 'expected_effect_en']

Nulls in search_unit_id (claims_reference): 0
